# 레이더 시계열 기반 이동 표적 식별 (Problem#1)
검증 환경: Python 3.9 / numpy 1.23.5 / pandas 1.5.2 / scikit-learn 1.1.3 (대회 서버 버전), CPU 2코어.

메인 버전 v2: 프레임·궤적 물리 피처 + ExtraTrees + LightGBM 확률 평균. LightGBM 설치 실패 시 sklearn `HistGradientBoosting`으로 자동 대체.

In [1]:
# 0. (필요 시) 추가 패키지 설치 - 실패해도 sklearn 대체 버전으로 진행
import importlib, subprocess, sys
try:
    importlib.import_module('lightgbm')
except ImportError:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'],
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
    print('lightgbm 설치', '성공' if r.returncode == 0 else '실패 -> sklearn HGB로 대체')
else:
    print('lightgbm 이미 설치됨')

lightgbm 설치 성공


In [2]:
# 1. 환경 및 설정
import os, sys, time, platform
import numpy as np, pandas as pd, sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
print('python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| sklearn', sklearn.__version__, '| lightgbm', lgb.__version__ if HAS_LGB else '없음 -> HGB 대체', '| cpu', os.cpu_count())

SEED = 42
N_JOBS = -1
key, tcol, target = 'sample_id', 't_ms', 'label'
CLASSES = ['HEAVY_VEHICLE', 'LIGHT_VEHICLE', 'PERSONNEL', 'PERSONNEL_GROUP', 'TWO_WHEELER']

# 데이터 경로 후보 (대회 서버 / 로컬 / Colab)
CANDIDATES = ['data', '.', '/content/drive/MyDrive/MAICON준비/2026/prob_01/data', '/content/data']
DATA_DIR = next(d for d in CANDIDATES if os.path.exists(os.path.join(d, 'train.csv')))
print('DATA_DIR =', DATA_DIR)

python 3.9.25 | numpy 1.23.5 | pandas 1.5.2 | sklearn 1.1.3 | lightgbm 4.6.0 | cpu 4
DATA_DIR = data


In [3]:
# 2. 데이터 로드 및 점검
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
print(train.shape, test.shape, train[key].nunique(), test[key].nunique())
print('결측', int(train.isnull().sum().sum()), int(test.isnull().sum().sum()))
assert (train.groupby(key)[target].nunique() == 1).all()   # sample 내 라벨 일관
print(train.groupby(key)[target].first().value_counts().to_dict())

(35776, 11) (10729, 10) 1000 300
결측 0 0
{'TWO_WHEELER': 200, 'PERSONNEL_GROUP': 200, 'PERSONNEL': 200, 'HEAVY_VEHICLE': 200, 'LIGHT_VEHICLE': 200}


In [4]:
# 3. 피처: sample 단위 통계 + 프레임(같은 t_ms) 단위 크기·미세도플러 + 중심 궤적
def q10(s): return s.quantile(0.1)
def q90(s): return s.quantile(0.9)

def make_X(sig):
    s = sig.copy().sort_values([key, tcol]).reset_index(drop=True)
    s['vabs'] = s['radial_velocity_comp_mps'].abs()
    s['p_lin'] = 10 ** (s['rcs_dbsm'] / 10.0)          # dBsm -> 선형 반사 면적
    g = s.groupby(key)
    # (1) 전체 detection 통계
    num = ['range_m', 'rcs_dbsm', 'radial_velocity_comp_mps', 'vabs', 'x_m', 'y_m', 'azimuth_rad']
    A = g[num].agg(['mean', 'std', 'min', 'max', 'median', q10, q90])
    A.columns = ['_'.join(c) for c in A.columns]
    A['n_det'] = g.size()
    A['n_frame'] = g[tcol].nunique()
    A['n_sensor'] = g['sensor_id'].nunique()
    A['dur'] = g[tcol].max() - g[tcol].min()
    A['det_per_frame'] = A['n_det'] / A['n_frame']
    A['log_n_det'] = np.log1p(A['n_det'])
    A['rcs_lin_sum_db'] = 10 * np.log10(g['p_lin'].sum())
    A['rcs_lin_mean_db'] = 10 * np.log10(g['p_lin'].mean())
    A['ego_comp_mean'] = (s['radial_velocity_mps'] - s['radial_velocity_comp_mps']).groupby(s[key]).mean()
    # (2) 프레임 단위: 표적 크기(점 퍼짐), 속도 퍼짐(미세도플러), 반사 강도
    fg = s.groupby([key, tcol])
    F = pd.DataFrame({
        'cnt': fg.size(),
        'xr': fg['x_m'].max() - fg['x_m'].min(),
        'yr': fg['y_m'].max() - fg['y_m'].min(),
        'rr': fg['range_m'].max() - fg['range_m'].min(),
        'vr': fg['radial_velocity_comp_mps'].max() - fg['radial_velocity_comp_mps'].min(),
        'vstd': fg['radial_velocity_comp_mps'].std(),
        'rcsmax': fg['rcs_dbsm'].max(),
        'rcssum': 10 * np.log10(fg['p_lin'].sum()),
        'rcsr': fg['rcs_dbsm'].max() - fg['rcs_dbsm'].min(),
        'cx': fg['x_m'].mean(), 'cy': fg['y_m'].mean(),
        'vmean': fg['radial_velocity_comp_mps'].mean(),
    }).reset_index()
    F['ext'] = np.sqrt(F['xr'] ** 2 + F['yr'] ** 2)
    fcols = ['cnt', 'xr', 'yr', 'rr', 'ext', 'vr', 'vstd', 'rcsmax', 'rcssum', 'rcsr']
    B = F.groupby(key)[fcols].agg(['mean', 'max', 'median', 'std'])
    B.columns = ['f_' + '_'.join(c) for c in B.columns]
    # (3) 프레임 중심 궤적의 선형 적합 -> 지상 속도, 잔차(흔들림)
    rows = {}
    for sid, d in F.groupby(key):
        t = d[tcol].values / 1000.0
        if len(t) >= 2 and t.max() > t.min():
            px = np.polyfit(t, d['cx'].values, 1); py = np.polyfit(t, d['cy'].values, 1)
            rx = d['cx'].values - np.polyval(px, t); ry = d['cy'].values - np.polyval(py, t)
            rows[sid] = (np.hypot(px[0], py[0]), np.sqrt(np.mean(rx ** 2 + ry ** 2)), np.std(d['vmean'].values))
        else:
            rows[sid] = (np.nan, np.nan, np.nan)
    T = pd.DataFrame.from_dict(rows, orient='index', columns=['trk_speed', 'trk_resid', 'frame_v_std'])
    X = A.join(B).join(T)
    X['spread_over_speed'] = X['f_vr_mean'] / (X['vabs_mean'] + 0.1)
    X['vstd_over_speed'] = X['radial_velocity_comp_mps_std'] / (X['vabs_mean'] + 0.1)
    X['detpf_x_range'] = X['det_per_frame'] * X['range_m_mean']
    X['rcs_plus_logr'] = X['rcs_lin_mean_db'] + 40 * np.log10(X['range_m_mean'] + 1)
    return X.replace([np.inf, -np.inf], np.nan)

t0 = time.time()
y = train.groupby(key)[target].first()
X = make_X(train).loc[y.index]                      # 정답과 같은 순서
X_test = make_X(test)
X_test = X_test.loc[sorted(test[key].unique()), X.columns]   # 학습과 같은 열 순서, sample_id 오름차순
assert X_test.shape[0] == test[key].nunique()
print(X.shape, X_test.shape, 'feature time %.1fs' % (time.time() - t0))

(1000, 105) (300, 105) feature time 13.9s


In [5]:
# 4. 모델 정의: ExtraTrees + (LightGBM 또는 HGB) 확률 평균
def make_models():
    et = ExtraTreesClassifier(n_estimators=800, random_state=SEED, n_jobs=N_JOBS)
    if HAS_LGB:
        gb = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, subsample=0.8, subsample_freq=1,
                                colsample_bytree=0.6, min_child_samples=10, random_state=SEED, n_jobs=N_JOBS, verbose=-1)
    else:
        gb = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_leaf_nodes=15,
                                            l2_regularization=1.0, random_state=SEED)
    return [('et', et, True), ('gb', gb, False)]     # (이름, 모델, NaN을 -999로 채울지)

def fit_predict(Xa, ya, Xb):
    P = np.zeros((len(Xb), len(CLASSES)))
    for name, m, fill in make_models():
        m.fit(Xa.fillna(-999) if fill else Xa, ya)
        assert list(m.classes_) == CLASSES
        P += m.predict_proba(Xb.fillna(-999) if fill else Xb)
    return P / 2

# 5. 5-fold Stratified CV (공식 지표 Accuracy)
t0 = time.time()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = np.zeros((len(X), len(CLASSES))); fold_acc = []
for k, (a, b) in enumerate(skf.split(X, y)):
    oof[b] = fit_predict(X.iloc[a], y.iloc[a], X.iloc[b])
    fold_acc.append(accuracy_score(y.iloc[b], np.array(CLASSES)[oof[b].argmax(1)]))
    print('fold', k, round(fold_acc[-1], 4))
oof_pred = np.array(CLASSES)[oof.argmax(1)]
print('CV mean %.4f ± %.4f | pooled OOF %.4f | %.0fs' % (np.mean(fold_acc), np.std(fold_acc), accuracy_score(y, oof_pred), time.time() - t0))
print(pd.DataFrame(confusion_matrix(y, oof_pred, labels=CLASSES), index=CLASSES, columns=CLASSES))

fold 0 0.885


fold 1 0.805


fold 2 0.805


fold 3 0.87


fold 4 0.895
CV mean 0.8520 ± 0.0392 | pooled OOF 0.8520 | 18s
                 HEAVY_VEHICLE  LIGHT_VEHICLE  PERSONNEL  PERSONNEL_GROUP  \
HEAVY_VEHICLE              176             21          0                1   
LIGHT_VEHICLE                3            175          3                7   
PERSONNEL                    0              0        168               28   
PERSONNEL_GROUP              0              1         40              156   
TWO_WHEELER                  0              7         10                6   

                 TWO_WHEELER  
HEAVY_VEHICLE              2  
LIGHT_VEHICLE             12  
PERSONNEL                  4  
PERSONNEL_GROUP            3  
TWO_WHEELER              177  


In [6]:
# 6. 전체 학습 -> test 예측 -> submission.csv
P_test = fit_predict(X, y, X_test)
sub = pd.DataFrame({key: X_test.index, target: np.array(CLASSES)[P_test.argmax(1)]})
sub = sub.sort_values(key).reset_index(drop=True)
sub.to_csv('submission.csv', index=False)

# 7. 제출 형식 검증 (저장 후 다시 읽기)
chk = pd.read_csv('submission.csv')
assert list(chk.columns) == [key, target]
assert len(chk) == test[key].nunique() and chk[key].is_unique
assert set(chk[key]) == set(test[key])
assert chk[key].tolist() == sorted(chk[key].tolist())
assert chk[target].isin(CLASSES).all() and chk[target].notnull().all()
print(chk.shape, chk.head())
print('train 비율', y.value_counts(normalize=True).sort_index().round(3).to_dict())
print('test 예측 비율', chk[target].value_counts(normalize=True).sort_index().round(3).to_dict())

(300, 2)    sample_id          label
0  VA_000001    TWO_WHEELER
1  VA_000002    TWO_WHEELER
2  VA_000003  LIGHT_VEHICLE
3  VA_000004  HEAVY_VEHICLE
4  VA_000005    TWO_WHEELER
train 비율 {'HEAVY_VEHICLE': 0.2, 'LIGHT_VEHICLE': 0.2, 'PERSONNEL': 0.2, 'PERSONNEL_GROUP': 0.2, 'TWO_WHEELER': 0.2}
test 예측 비율 {'HEAVY_VEHICLE': 0.17, 'LIGHT_VEHICLE': 0.207, 'PERSONNEL': 0.207, 'PERSONNEL_GROUP': 0.237, 'TWO_WHEELER': 0.18}
